In [1]:
!pip install transformers accelerate soundfile langdetect torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 59.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=ed8db13f36ec53531016f0786f26b8a7dfa80e25af56495d076ecf7fc8abd9cc
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
#!/usr/bin/env python3
"""Task 1.2: Constrained Decoding with N-gram Logit Bias.

"""

from __future__ import annotations

import argparse
import gc
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import soundfile as sf
import torch
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    LogitsProcessor,
    LogitsProcessorList,
)

TARGET_SR = 16_000


In [9]:
# ══════════════════════════════════════════════════════
#  N-gram Language Model
# ══════════════════════════════════════════════════════
class NgramLM:
    """Simple N-gram language model with add-k smoothing."""

    def __init__(self, order: int = 3, smoothing_k: float = 0.01):
            self.order = order
            self.smoothing_k = smoothing_k
            self.ngram_counts: Dict[Tuple[str, ...], Counter] = defaultdict(Counter)
            self.vocab: set = set()

    def train(self, corpus_path: str) -> None:
            """Train N-gram model from a text corpus file."""
            text = Path(corpus_path).read_text(encoding="utf-8")
            # Tokenise: lowercase words
            words = re.findall(r"[a-zA-Z]+", text.lower())
            self.vocab = set(words)

            for n in range(1, self.order + 1):
                for i in range(len(words) - n):
                    context = tuple(words[i : i + n])
                    next_word = words[i + n] if i + n < len(words) else "<eos>"
                    self.ngram_counts[context][next_word] += 1

            print(f"  N-gram LM trained: order={self.order}, vocab={len(self.vocab)}, "
                  f"unique contexts={len(self.ngram_counts)}")

    def score(self, context_words: List[str], candidate: str) -> float:
            """Log-probability of candidate given context (with backoff)."""
            context_words = [w.lower() for w in context_words]
            candidate = candidate.lower()

            for n in range(min(self.order, len(context_words)), 0, -1):
                ctx = tuple(context_words[-n:])
                if ctx in self.ngram_counts:
                    counts = self.ngram_counts[ctx]
                    total = sum(counts.values())
                    count = counts.get(candidate, 0)
                    prob = (count + self.smoothing_k) / (total + self.smoothing_k * max(len(self.vocab), 1))
                    return math.log(prob + 1e-30)

            # Unigram fallback
            return -10.0  # very low score for unknown context

    def get_technical_terms(self) -> set:
            """Return the set of technical vocabulary words from the corpus."""
            return self.vocab.copy()


# ══════════════════════════════════════════════════════
#  Whisper Logit Bias Processor
# ══════════════════════════════════════════════════════
class SyllabusLogitBiasProcessor(LogitsProcessor):
    """Applies a logit bias to boost tokens that are part of technical terms."""

    def __init__(
        self,
        processor: WhisperProcessor,
        ngram_lm: NgramLM,
        bias_weight: float = 2.5,
    ):
        self.tokenizer = processor.tokenizer
        self.ngram_lm = ngram_lm
        self.bias_weight = bias_weight

        # Pre-compute token → term mapping for technical vocabulary
        self.term_token_ids: Dict[int, str] = {}
        for term in ngram_lm.get_technical_terms():
            token_ids = self.tokenizer.encode(term, add_special_tokens=False)
            for tid in token_ids:
                decoded = self.tokenizer.decode([tid]).strip().lower()
                if len(decoded) >= 2:  # skip single-char noise
                    self.term_token_ids[tid] = decoded

        print(f"  Logit bias covers {len(self.term_token_ids)} token IDs "
              f"from {len(ngram_lm.get_technical_terms())} terms")

    def __call__(
        self, input_ids: torch.LongTensor, scores: torch.FloatTensor
    ) -> torch.FloatTensor:
        # Decode recent tokens as context for N-gram scoring
        batch_size = input_ids.shape[0]
        for b in range(batch_size):
            recent = input_ids[b, -self.ngram_lm.order:]
            context_text = self.tokenizer.decode(recent, skip_special_tokens=True)
            context_words = re.findall(r"[a-zA-Z]+", context_text.lower())

            for tid, decoded_term in self.term_token_ids.items():
                ngram_score = self.ngram_lm.score(context_words, decoded_term)
                # Only boost, never penalise
                if ngram_score > -8.0:
                    scores[b, tid] += self.bias_weight * max(0, ngram_score + 10.0)

        return scores


# ══════════════════════════════════════════════════════
#  Audio loading
# ══════════════════════════════════════════════════════
def resolve_audio_path(path_str: str) -> Path:
    path = Path(path_str)
    if path.exists():
        return path
    fallback = Path("input") / path_str
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f"Audio not found: '{path}' and fallback '{fallback}'")


def load_mono_16k(path: Path) -> np.ndarray:
    wav_np, sr = sf.read(str(path), always_2d=True, dtype="float32")
    if wav_np.size == 0:
        raise ValueError(f"Empty audio file: {path}")
    wav = wav_np.mean(axis=1) if wav_np.shape[1] > 1 else wav_np[:, 0]

    if sr != TARGET_SR:
        import torchaudio
        t = torch.from_numpy(wav).unsqueeze(0)
        t = torchaudio.functional.resample(t, sr, TARGET_SR)
        wav = t.squeeze(0).numpy()

    return wav


# ══════════════════════════════════════════════════════
#  Chunked transcription
# ══════════════════════════════════════════════════════
def transcribe_chunked(
    audio: np.ndarray,
    processor: WhisperProcessor,
    model: WhisperForConditionalGeneration,
    logits_processor: LogitsProcessorList,
    device: torch.device,
    chunk_s: float = 30.0,
    stride_s: float = 5.0,
) -> List[dict]:
    """Transcribe audio in overlapping chunks and merge results."""
    chunk_samples = int(chunk_s * TARGET_SR)
    stride_samples = int(stride_s * TARGET_SR)
    total_samples = len(audio)

    all_segments: List[dict] = []
    start = 0
    chunk_idx = 0

    while start < total_samples:
        end = min(start + chunk_samples, total_samples)
        chunk = audio[start:end]

        if len(chunk) < TARGET_SR:  # skip <1s tail
            break

        # Prepare input features
        input_features = processor(
            chunk, sampling_rate=TARGET_SR, return_tensors="pt"
        ).input_features.to(device, dtype=torch.float16 if device.type == "cuda" else torch.float32)

        # Generate with constrained decoding
        with torch.no_grad():
            generated_ids = model.generate(
                input_features,
                logits_processor=logits_processor,
                return_timestamps=True,
                max_new_tokens=256,
            )

        # Decode result
        result = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )

        chunk_start_s = start / TARGET_SR
        text = result[0] if len(result) > 0 else ""

        all_segments.append({
            "start_s": round(chunk_start_s, 2),
            "end_s": round(end / TARGET_SR, 2),
            "text": text.strip(),
        })

        chunk_idx += 1
        if chunk_idx % 5 == 0:
            print(f"  Processed chunk {chunk_idx} (up to {end/TARGET_SR:.1f}s)")
        gc.collect()

        # Advance by chunk minus stride (overlap)
        start += chunk_samples - stride_samples

    return all_segments


# ══════════════════════════════════════════════════════
#  CLI
# ══════════════════════════════════════════════════════
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Task 1.2: Constrained Decoding with N-gram logit bias"
    )
    parser.add_argument(
        "--audio",
        default="denoised_segment.wav",
        help="Input audio file (default: denoised_segment.wav).",
    )
    parser.add_argument(
        "--corpus",
        default="syllabus_corpus.txt",
        help="Syllabus corpus for N-gram LM (default: syllabus_corpus.txt).",
    )
    parser.add_argument(
        "--output",
        default="transcript_constrained.json",
        help="Output transcript JSON path.",
    )
    parser.add_argument(
        "--whisper-model",
        default="openai/whisper-large-v3",
        help="Whisper model id (default: whisper-large-v3 for Colab).",
    )
    parser.add_argument(
        "--ngram-order", type=int, default=3,
        help="N-gram order (default: 3).",
    )
    parser.add_argument(
        "--bias-weight", type=float, default=2.5,
        help="Logit bias weight for technical terms (default: 2.5).",
    )
    parser.add_argument(
        "--chunk-seconds", type=float, default=30.0,
        help="Chunk length in seconds for processing.",
    )
    return parser.parse_args()

In [10]:

class Args: pass
args = Args()
args.bootstrap_audio = "original_segment.wav"
args.inference_audio = "denoised_segment.wav"
args.audio = "original_segment.wav" # for task 1.2
args.corpus = "syllabus_corpus.txt"
args.output = "transcript_constrained.json"
args.output_json = "lid_labels.json"
args.whisper_model = "openai/whisper-large-v3"
args.wav2vec_model = "facebook/wav2vec2-base"
args.ngram_order = 3
args.bias_weight = 2.5
args.epochs = 10
args.batch_size = 512
args.lr = 1e-3
args.seed = 42
args.chunk_seconds = 30.0
args.min_switch_ms = 200.0
args.save_model = "lid_model.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Step 1: Train N-gram LM from syllabus ──

Using device: cuda


In [11]:
print("\n[Step 1/3] Training N-gram language model from syllabus...")
ngram_lm = NgramLM(order=args.ngram_order)
corpus_path = resolve_audio_path(args.corpus) if Path(args.corpus).exists() else Path(args.corpus)
ngram_lm.train(str(corpus_path))

# ── Step 2: Load Whisper + create logits processor ──


[Step 1/3] Training N-gram language model from syllabus...
  N-gram LM trained: order=3, vocab=265, unique contexts=1180


In [12]:
print("\n[Step 2/3] Loading Whisper model...")
processor = WhisperProcessor.from_pretrained(args.whisper_model)
model = WhisperForConditionalGeneration.from_pretrained(
    args.whisper_model, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32
).to(device)
model.eval()

bias_processor = SyllabusLogitBiasProcessor(
    processor=processor,
    ngram_lm=ngram_lm,
    bias_weight=args.bias_weight,
)
logits_processor = LogitsProcessorList([bias_processor])

# ── Step 3: Transcribe ──


[Step 2/3] Loading Whisper model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

  Logit bias covers 370 token IDs from 265 terms


In [13]:
print("\n[Step 3/3] Transcribing with constrained decoding...")
try:
    audio_path = resolve_audio_path(args.audio)
except FileNotFoundError:
    print(f"  Warning: '{args.audio}' not found, using original audio.")
    audio_path = resolve_audio_path("original_segment.wav")

audio = load_mono_16k(audio_path)
print(f"  Audio: {len(audio)/TARGET_SR:.1f}s, file: {audio_path}")

segments = transcribe_chunked(
    audio=audio,
    processor=processor,
    model=model,
    logits_processor=logits_processor,
    device=device,
    chunk_s=args.chunk_seconds,
)

# Combine full transcript
full_text = " ".join(seg["text"] for seg in segments if seg["text"])

output = {
    "model": args.whisper_model,
    "ngram_order": args.ngram_order,
    "bias_weight": args.bias_weight,
    "audio_file": str(audio_path),
    "segments": segments,
    "full_transcript": full_text,
}

with open(args.output, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\n  Saved transcript ({len(segments)} segments) to: {args.output}")
print(f"  Full transcript length: {len(full_text)} chars")

# Free memory
del model, processor
gc.collect()
print("  Done!")



[Step 3/3] Transcribing with constrained decoding...
  Audio: 599.6s, file: original_segment.wav


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logi

  Processed chunk 5 (up to 130.0s)


Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processed chunk 10 (up to 255.0s)


...

  Processed chunk 15 (up to 380.0s)


Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Processed chunk 20 (up to 505.0s)


Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  Saved transcript (24 segments) to: transcript_constrained.json
  Full transcript length: 8166 chars
  Done!
